# IS3107 Chess ML Model

This notebook is organized as a simple pipeline:

1. Install dependencies
2. Configure the run
3. Define helper functions
4. Download and parse Lichess PGN data
5. Inspect missing evaluation coverage
6. Fill missing evaluations with Stockfish
7. Inspect the filled dataset before feature engineering

Notes:

- Existing Lichess `%eval` annotations are preserved.
- Stockfish is only used for rows where both `eval_cp` and `mate_in` are missing.
- Full-engine scoring on every missing row can be expensive. Start with smaller settings, validate, then scale up.
        


In [13]:
!pip install requests zstandard python-chess pandas


In [1]:
# Run configuration
BROADCAST_URLS = [
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2026-03.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2026-02.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2026-01.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2025-12.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2025-11.pgn.zst",
    "https://database.lichess.org/broadcast/lichess_db_broadcast_2025-10.pgn.zst",
]

# Filter config
TARGET_FILTERED_GAMES = 2000
MIN_ELO = 2700
MAX_GAME_AGE_DAYS = 730
MIN_TIME_CONTROL_SECONDS = 15 * 60

# Engine-eval config
FILL_MISSING_EVALS = True
STOCKFISH_PATH = None  # Set this manually if stockfish is not on PATH.
ENGINE_DEPTH = 1       # Use shallow depth for scalable feature generation.
ENGINE_TIME_LIMIT = 0.01
ENGINE_MAX_ROWS = None  # Set to an int for a quick test run.
        


In [3]:
import io
import os
import re
import shutil
from typing import Dict, List, Optional

import chess
import chess.engine
import chess.pgn
import pandas as pd
import requests
import zstandard as zstd
from tqdm import tqdm


DEFAULT_STOCKFISH_CANDIDATES = [
    STOCKFISH_PATH,
    os.environ.get("STOCKFISH_PATH"),
    shutil.which("stockfish"),
    "/opt/homebrew/bin/stockfish",
    "/usr/local/bin/stockfish",
]


def stream_text_from_zst_url(url: str) -> io.TextIOBase:
    resp = requests.get(url, stream=True, timeout=120)
    resp.raise_for_status()

    dctx = zstd.ZstdDecompressor()
    reader = dctx.stream_reader(resp.raw)
    return io.TextIOWrapper(reader, encoding="utf-8", errors="replace")


def parse_clock_to_seconds(clock_str: str) -> Optional[float]:
    parts = clock_str.split(":")
    try:
        parts = [float(x) for x in parts]
    except ValueError:
        return None

    if len(parts) == 3:
        h, m, s = parts
        return h * 3600 + m * 60 + s
    if len(parts) == 2:
        m, s = parts
        return m * 60 + s
    if len(parts) == 1:
        return parts[0]
    return None


def parse_lichess_date(date_str: Optional[str]) -> Optional[pd.Timestamp]:
    if not date_str or date_str in {"????.??.??", "????.??.??"}:
        return None
    try:
        return pd.to_datetime(date_str, format="%Y.%m.%d", errors="raise").normalize()
    except (TypeError, ValueError):
        return None


def parse_time_control_base_seconds(time_control: Optional[str]) -> Optional[int]:
    if not time_control or time_control in {"-", "?"}:
        return None

    base_str = time_control.split("+", 1)[0]
    if not base_str.isdigit():
        return None
    return int(base_str)


def game_matches_filters(
    headers,
    min_elo: int,
    min_time_control_seconds: int,
    earliest_date: pd.Timestamp,
    latest_date: pd.Timestamp,
) -> bool:
    white_elo_raw = headers.get("WhiteElo")
    black_elo_raw = headers.get("BlackElo")
    white_elo = int(white_elo_raw) if white_elo_raw and white_elo_raw.isdigit() else None
    black_elo = int(black_elo_raw) if black_elo_raw and black_elo_raw.isdigit() else None
    has_super_gm = (white_elo is not None and white_elo >= min_elo) or (black_elo is not None and black_elo >= min_elo)

    game_date = parse_lichess_date(headers.get("Date"))
    if game_date is None:
        return False

    base_seconds = parse_time_control_base_seconds(headers.get("TimeControl"))
    if base_seconds is None:
        return False

    is_recent_enough = earliest_date <= game_date <= latest_date
    is_long_time_control = base_seconds >= min_time_control_seconds
    return has_super_gm and is_recent_enough and is_long_time_control


def extract_eval_and_clock(comment: str):
    eval_cp = None
    mate_in = None
    clk_seconds = None

    eval_match = re.search(r"\[%eval\s+([^\]]+)\]", comment)
    clk_match = re.search(r"\[%clk\s+([^\]]+)\]", comment)

    if eval_match:
        raw_eval = eval_match.group(1).strip()
        if raw_eval.startswith("#"):
            try:
                mate_in = int(raw_eval[1:])
            except ValueError:
                mate_in = None
        else:
            try:
                eval_cp = int(round(float(raw_eval) * 100))
            except ValueError:
                eval_cp = None

    if clk_match:
        clk_seconds = parse_clock_to_seconds(clk_match.group(1).strip())

    return eval_cp, mate_in, clk_seconds


def result_to_label(result: str) -> str:
    if result == "1-0":
        return "white_win"
    if result == "0-1":
        return "black_win"
    if result == "1/2-1/2":
        return "draw"
    return "unknown"


def find_stockfish_binary(stockfish_path: Optional[str] = None) -> str:
    candidates = [stockfish_path, *DEFAULT_STOCKFISH_CANDIDATES]
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate

    raise FileNotFoundError(
        "Stockfish binary not found. Install Stockfish locally and set STOCKFISH_PATH if needed."
    )


def score_fen_with_stockfish(
    engine: chess.engine.SimpleEngine,
    fen: str,
    depth: int = 1,
    time_limit: Optional[float] = 0.01,
):
    board = chess.Board(fen)
    limit = chess.engine.Limit(depth=depth) if time_limit is None else chess.engine.Limit(depth=depth, time=time_limit)
    info = engine.analyse(board, limit)
    score = info["score"].white()

    if score.is_mate():
        return None, score.mate()
    return score.score(mate_score=100000), None


def fill_missing_evals_with_stockfish(
    df: pd.DataFrame,
    stockfish_path: Optional[str] = None,
    depth: int = 1,
    time_limit: Optional[float] = 0.01,
    max_rows: Optional[int] = None,
) -> pd.DataFrame:
    missing_mask = df["eval_cp"].isna() & df["mate_in"].isna()
    missing_idx = list(df.index[missing_mask])

    if max_rows is not None:
        missing_idx = missing_idx[:max_rows]

    if not missing_idx:
        print("No missing evals to fill.")
        return df

    engine_path = find_stockfish_binary(stockfish_path)
    print(f"Using Stockfish at: {engine_path}")
    print(f"Rows queued for engine eval: {len(missing_idx)}")

    filled_df = df.copy()
    fen_cache = {}

    with chess.engine.SimpleEngine.popen_uci(engine_path) as engine:
        for idx in tqdm(missing_idx, desc="Computing missing evals", unit="row"):
            fen = filled_df.at[idx, "fen_after"]
            if fen not in fen_cache:
                fen_cache[fen] = score_fen_with_stockfish(
                    engine,
                    fen,
                    depth=depth,
                    time_limit=time_limit,
                )
            eval_cp, mate_in = fen_cache[fen]
            filled_df.at[idx, "eval_cp"] = eval_cp
            filled_df.at[idx, "mate_in"] = mate_in

    return filled_df


def load_filtered_broadcast_games(
    target_games: int = 1000,
    min_elo: int = 2700,
    max_game_age_days: int = 730,
    min_time_control_seconds: int = 15 * 60,
    urls: Optional[List[str]] = None,
) -> pd.DataFrame:
    if urls is None:
        urls = BROADCAST_URLS

    rows: List[Dict] = []
    today = pd.Timestamp.today().normalize()
    earliest_date = today - pd.Timedelta(days=max_game_age_days)
    matched_games = 0

    with tqdm(total=target_games, desc="Parsing filtered games", unit="game") as pbar:
        for url in urls:
            if matched_games >= target_games:
                break

            print(f"Reading from: {url}")
            text_stream = stream_text_from_zst_url(url)

            while matched_games < target_games:
                game = chess.pgn.read_game(text_stream)
                if game is None:
                    break

                headers = game.headers
                if not game_matches_filters(
                    headers,
                    min_elo=min_elo,
                    min_time_control_seconds=min_time_control_seconds,
                    earliest_date=earliest_date,
                    latest_date=today,
                ):
                    continue

                board = game.board()

                result = headers.get("Result", "")
                white_elo = headers.get("WhiteElo")
                black_elo = headers.get("BlackElo")
                site = headers.get("Site", "")
                game_id = site.rstrip("/").split("/")[-1] if site else None

                node = game
                ply = 0

                while node.variations:
                    next_node = node.variation(0)
                    move = next_node.move
                    ply += 1

                    fen_before = board.fen()
                    side_to_move = "white" if board.turn == chess.WHITE else "black"
                    san = board.san(move)
                    uci = move.uci()

                    board.push(move)
                    fen_after = board.fen()

                    comment = next_node.comment or ""
                    eval_cp, mate_in, clk_seconds = extract_eval_and_clock(comment)

                    rows.append(
                        {
                            "game_id": game_id,
                            "date": headers.get("Date"),
                            "white_player": headers.get("White"),
                            "black_player": headers.get("Black"),
                            "white_elo": int(white_elo) if white_elo and white_elo.isdigit() else None,
                            "black_elo": int(black_elo) if black_elo and black_elo.isdigit() else None,
                            "result": result,
                            "label": result_to_label(result),
                            "time_control": headers.get("TimeControl"),
                            "eco": headers.get("ECO"),
                            "opening": headers.get("Opening"),
                            "ply": ply,
                            "side_to_move": side_to_move,
                            "san": san,
                            "uci": uci,
                            "fen_before": fen_before,
                            "fen_after": fen_after,
                            "eval_cp": eval_cp,
                            "mate_in": mate_in,
                            "clock_seconds_after_move": clk_seconds,
                        }
                    )

                    node = next_node

                matched_games += 1
                pbar.update(1)

    return pd.DataFrame(rows)
        


In [4]:
# Step 1: Download and parse the dataset
df = load_filtered_broadcast_games(
    target_games=TARGET_FILTERED_GAMES,
    min_elo=MIN_ELO,
    max_game_age_days=MAX_GAME_AGE_DAYS,
    min_time_control_seconds=MIN_TIME_CONTROL_SECONDS,
)

print(df.head())
print("Rows:", len(df))
print("Unique games:", df["game_id"].nunique())
print("Estimated rows target:", TARGET_FILTERED_GAMES * 200)
        


Parsing filtered games:   0%|                                                                      | 0/2000 [00:00<?, ?game/s]

Reading from: https://database.lichess.org/broadcast/lichess_db_broadcast_2026-03.pgn.zst


Parsing filtered games:  12%|███████▍                                                    | 246/2000 [00:47<04:18,  6.79game/s]

Reading from: https://database.lichess.org/broadcast/lichess_db_broadcast_2026-02.pgn.zst


Parsing filtered games:  41%|████████████████████████▌                                   | 820/2000 [01:45<02:22,  8.31game/s]

...

Reading from: https://database.lichess.org/broadcast/lichess_db_broadcast_2025-12.pgn.zst


Parsing filtered games:  94%|███████████████████████████████████████████████████████▋   | 1886/2000 [04:07<00:10, 10.44game/s]

Reading from: https://database.lichess.org/broadcast/lichess_db_broadcast_2025-11.pgn.zst


illegal san: '0-0' in rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 while parsing <Game at 0x16b6effe0 ('Martinez Gonzalez, Miguel A.' vs. 'Ramirez Maillo, Cristian', '2025.11.02' at 'https://lichess.org/broadcast/benidorm-chess-open-2025--group-a/round-9/zqiflA9E/ptwjkSg1')>
Parsing filtered games: 100%|███████████████████████████████████████████████████████████| 2000/2000 [04:52<00:00,  6.83game/s]


    game_id        date      white_player        black_player  white_elo  \
0  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   
1  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   
2  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   
3  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   
4  wjVeV6zD  2026.01.20  Berserk 20250622  Obsidian dev-16.15       3667   

   black_elo   result label time_control  eco  \
0       3714  1/2-1/2  draw       3600+6  B27   
1       3714  1/2-1/2  draw       3600+6  B27   
2       3714  1/2-1/2  draw       3600+6  B27   
3       3714  1/2-1/2  draw       3600+6  B27   
4       3714  1/2-1/2  draw       3600+6  B27   

                                     opening  ply side_to_move  san   uci  \
0  Sicilian Defense: Hyperaccelerated Dragon    1        white  Nf3  g1f3   
1  Sicilian Defense: Hyperaccelerated Dragon    2        black   g6  g7g6   
2  Sicilian Defen

## Step 5: Feature Engineering
We keep feature engineering explicit and interpretable. Missing `eval_cp` values are filled first, then the remaining modeling features are created one by one.


### 5.1 Fill Missing `eval_cp`


In [37]:
import numpy as np

model_df = df.copy()

total_rows = len(model_df)
eval_cp_missing_before = model_df["eval_cp"].isna().sum()
mate_in_non_null_before = model_df["mate_in"].notna().sum()
rows_with_any_eval_before = (model_df["eval_cp"].notna() | model_df["mate_in"].notna()).sum()
rows_without_any_eval_before = total_rows - rows_with_any_eval_before

if FILL_MISSING_EVALS:
    model_df = fill_missing_evals_with_stockfish(
        model_df,
        stockfish_path=STOCKFISH_PATH,
        depth=ENGINE_DEPTH,
        time_limit=ENGINE_TIME_LIMIT,
        max_rows=ENGINE_MAX_ROWS,
    )
else:
    print("Skipping Stockfish fill.")



Using Stockfish at: /opt/homebrew/bin/stockfish
Rows queued for engine eval: 34864


Computing missing evals: 100%|███████████████████████████████████████████████████████| 34864/34864 [00:26<00:00, 1311.35row/s]


### Validate `eval_cp` Fill


In [ ]:
# `eval_cp` and `mate_in` both come from the engine's evaluation of the current position.
# If the position is still being judged in centipawns, the signal appears in `eval_cp`.
# If the engine sees a forced checkmate sequence, it may report that as `mate_in` instead.
# In that sense, `mate_in` is not a completely separate concept from `eval_cp`; it is a stronger,
# more decisive version of engine evaluation for positions where mate is already forced.
# For validation, we therefore check whether a row has either `eval_cp` or `mate_in`, because
# either one means the row contains some engine-based signal about the position.

In [39]:
eval_cp_missing_after = model_df["eval_cp"].isna().sum()
mate_in_non_null_after = model_df["mate_in"].notna().sum()
rows_with_any_eval_after = (model_df["eval_cp"].notna() | model_df["mate_in"].notna()).sum()
rows_without_any_eval_after = total_rows - rows_with_any_eval_after
filled_eval_rows = eval_cp_missing_before - eval_cp_missing_after

print("=== eval_cp fill validation ===")
print("Total rows:", total_rows)
print("Rows without eval_cp and mate_in before fill:", rows_without_any_eval_before)
print("Rows without eval_cp and mate_in after fill:", rows_without_any_eval_after)
print("Rows without eval_cp before fill:", eval_cp_missing_before)
print("Rows without eval_cp after fill:", eval_cp_missing_after)
print("New eval_cp values filled:", filled_eval_rows)
print("Rows with mate_in before fill:", mate_in_non_null_before)
print("Rows with mate_in after fill:", mate_in_non_null_after)
print("Rows with any eval signal before fill:", rows_with_any_eval_before)
print("Rows with any eval signal after fill:", rows_with_any_eval_after)
print("Fraction with any eval signal after fill:", rows_with_any_eval_after / total_rows if total_rows else 0)

eval_series = model_df["eval_cp"].dropna()
print("=== eval_cp stats after fill ===")
print("min:", eval_series.min())
print("max:", eval_series.max())
print("positive:", (eval_series > 0).sum())
print("negative:", (eval_series < 0).sum())
print("zero:", (eval_series == 0).sum())

model_df[model_df["eval_cp"].notna() | model_df["mate_in"].notna()].head(10)



=== eval_cp fill validation ===
Total rows: 337545
Rows without eval_cp and mate_in before fill: 34864
Rows without eval_cp and mate_in after fill: 0
Rows without eval_cp before fill: 56063
Rows without eval_cp after fill: 22259
New eval_cp values filled: 33804
Rows with mate_in before fill: 21199
Rows with mate_in after fill: 22259
Rows with any eval signal before fill: 302681
Rows with any eval signal after fill: 337545
Fraction with any eval signal after fill: 1.0
=== eval_cp stats after fill ===
min: -8325.0
max: 19908.0
positive: 271002
negative: 16998
zero: 27286


,game_id,date,white_player,black_player,white_elo,black_elo,result,label,time_control,eco,opening,ply,side_to_move,san,uci,fen_before,fen_after,eval_cp,mate_in,clock_seconds_after_move
0,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,1,white,Nf3,g1f3,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,rnbqkbnr/pppppppp/8/8/8/5N2/PPPPPPPP/RNBQKB1R ...,10.0,NaN,3600.0
1,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,2,black,g6,g7g6,rnbqkbnr/pppppppp/8/8/8/5N2/PPPPPPPP/RNBQKB1R ...,rnbqkbnr/pppppp1p/6p1/8/8/5N2/PPPPPPPP/RNBQKB1...,38.0,NaN,3600.0
2,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,3,white,e4,e2e4,rnbqkbnr/pppppp1p/6p1/8/8/5N2/PPPPPPPP/RNBQKB1...,rnbqkbnr/pppppp1p/6p1/8/4P3/5N2/PPPP1PPP/RNBQK...,51.0,NaN,3600.0
3,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,4,black,c5,c7c5,rnbqkbnr/pppppp1p/6p1/8/4P3/5N2/PPPP1PPP/RNBQK...,rnbqkbnr/pp1ppp1p/6p1/2p5/4P3/5N2/PPPP1PPP/RNB...,34.0,NaN,3600.0
4,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,5,white,c3,c2c3,rnbqkbnr/pp1ppp1p/6p1/2p5/4P3/5N2/PPPP1PPP/RNB...,rnbqkbnr/pp1ppp1p/6p1/2p5/4P3/2P2N2/PP1P1PPP/R...,24.0,NaN,3600.0
5,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,6,black,Nf6,g8f6,rnbqkbnr/pp1ppp1p/6p1/2p5/4P3/2P2N2/PP1P1PPP/R...,rnbqkb1r/pp1ppp1p/5np1/2p5/4P3/2P2N2/PP1P1PPP/...,49.0,NaN,3600.0
6,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,7,white,e5,e4e5,rnbqkb1r/pp1ppp1p/5np1/2p5/4P3/2P2N2/PP1P1PPP/...,rnbqkb1r/pp1ppp1p/5np1/2p1P3/8/2P2N2/PP1P1PPP/...,48.0,NaN,3600.0
7,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,8,black,Nd5,f6d5,rnbqkb1r/pp1ppp1p/5np1/2p1P3/8/2P2N2/PP1P1PPP/...,rnbqkb1r/pp1ppp1p/6p1/2pnP3/8/2P2N2/PP1P1PPP/R...,53.0,NaN,3600.0
8,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,9,white,d4,d2d4,rnbqkb1r/pp1ppp1p/6p1/2pnP3/8/2P2N2/PP1P1PPP/R...,rnbqkb1r/pp1ppp1p/6p1/2pnP3/3P4/2P2N2/PP3PPP/R...,47.0,NaN,3600.0
9,wjVeV6zD,2026.01.20,Berserk 20250622,Obsidian dev-16.15,3667,3714,1/2-1/2,draw,3600+6,B27,Sicilian Defense: Hyperaccelerated Dragon,10,black,cxd4,c5d4,rnbqkb1r/pp1ppp1p/6p1/2pnP3/3P4/2P2N2/PP3PPP/R...,rnbqkb1r/pp1ppp1p/6p1/3nP3/3p4/2P2N2/PP3PPP/RN...,44.0,NaN,3600.0


### 5.2 Elo Difference Feature


In [41]:
# Elo difference captures the overall skill gap between White and Black.
model_df["elo_diff"] = model_df["white_elo"] - model_df["black_elo"]

model_df[["white_elo", "black_elo", "elo_diff"]].head(10)



,white_elo,black_elo,elo_diff
0,3667,3714,-47
1,3667,3714,-47
2,3667,3714,-47
3,3667,3714,-47
4,3667,3714,-47
5,3667,3714,-47
6,3667,3714,-47
7,3667,3714,-47
8,3667,3714,-47
9,3667,3714,-47


### 5.3 Time Left Feature


In [43]:
# Remaining clock time captures time pressure after the move was played.
model_df["time_left_seconds"] = model_df["clock_seconds_after_move"].clip(lower=0)

model_df[["clock_seconds_after_move", "time_left_seconds"]].head(10)



,clock_seconds_after_move,time_left_seconds
0,3600.0,3600.0
1,3600.0,3600.0
2,3600.0,3600.0
3,3600.0,3600.0
4,3600.0,3600.0
5,3600.0,3600.0
6,3600.0,3600.0
7,3600.0,3600.0
8,3600.0,3600.0
9,3600.0,3600.0


### 5.4 Position Context Features


In [45]:
# Engine evaluation is the strongest signal of which side is better in the current position.
model_df["eval_cp_clipped"] = model_df["eval_cp"].clip(-1000, 1000)
# Ply tells the model how early or late the position is in the game.
model_df["ply"] = model_df["ply"].astype(int)
# Side to move tells the model whose turn it is in the current position.
model_df["side_to_move"] = model_df["side_to_move"].astype("category")

model_df[["eval_cp", "eval_cp_clipped", "ply", "side_to_move"]].head(10)



,eval_cp,eval_cp_clipped,ply,side_to_move
0,10.0,10.0,1,white
1,38.0,38.0,2,black
2,51.0,51.0,3,white
3,34.0,34.0,4,black
4,24.0,24.0,5,white
5,49.0,49.0,6,black
6,48.0,48.0,7,white
7,53.0,53.0,8,black
8,47.0,47.0,9,white
9,44.0,44.0,10,black


### 5.5 Final Modeling Table


In [47]:
required_columns = [
    "game_id",
    "label",
    "eval_cp_clipped",
    "white_elo",
    "black_elo",
    "elo_diff",
    "time_left_seconds",
    "ply",
    "side_to_move",
]

modeling_input_df = model_df.copy()

model_df = model_df.dropna(subset=required_columns).copy()

feature_cols = [
    "eval_cp_clipped",
    "elo_diff",
    "time_left_seconds",
    "ply",
    "side_to_move",
]

label_order = ["white_win", "draw", "black_win"]
label_to_id = {label: idx for idx, label in enumerate(label_order)}
model_df = model_df[model_df["label"].isin(label_to_id)].copy()
model_df["target"] = model_df["label"].map(label_to_id).astype(int)

print("Rows kept for modeling:", len(model_df))
print("Unique games kept:", model_df["game_id"].nunique())
print("Feature columns:", feature_cols)
print(model_df[["label", "target"]].drop_duplicates().sort_values("target"))



Rows kept for modeling: 314586
Unique games kept: 1997
Feature columns: ['eval_cp_clipped', 'elo_diff', 'time_left_seconds', 'ply', 'side_to_move']
           label  target
122    white_win       0
0           draw       1
51635  black_win       2


## Step 6: Train / Validation / Test Split
We split by `game_id` instead of by row so that move rows from the same game do not leak across datasets.


In [ ]:
from sklearn.model_selection import train_test_split

game_level_df = model_df.groupby("game_id", as_index=False).agg(game_label=("label", "first"))

train_val_games, test_games = train_test_split(
    game_level_df,
    test_size=0.15,
    random_state=42,
    stratify=game_level_df["game_label"],
)

val_size_within_train_val = 0.15 / 0.85
train_games, val_games = train_test_split(
    train_val_games,
    test_size=val_size_within_train_val,
    random_state=42,
    stratify=train_val_games["game_label"],
)

train_ids = set(train_games["game_id"])
val_ids = set(val_games["game_id"])
test_ids = set(test_games["game_id"])

train_df = model_df[model_df["game_id"].isin(train_ids)].copy()
val_df = model_df[model_df["game_id"].isin(val_ids)].copy()
test_df = model_df[model_df["game_id"].isin(test_ids)].copy()

print("Train rows:", len(train_df), "| games:", train_df["game_id"].nunique())
print("Val rows:", len(val_df), "| games:", val_df["game_id"].nunique())
print("Test rows:", len(test_df), "| games:", test_df["game_id"].nunique())

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
